# Testing Notebook

This notebook **loads a saved pipeline** produced by the training notebook and evaluates it on the held-out test set. It expects the `trained_models/` folder to contain:

- `best_pipeline_<model>_*.joblib` (or update `MODEL_FILE` in Cell 2)
- `X_test.csv` and `y_test.csv`
- `training_metadata.json` (optional)

Created: 2025-10-09T21:24:21.510253Z


In [ ]:
# Cell 1: Environment versions & quick checks
import sys, platform, sklearn, pandas as pd, numpy as np
print('python:', sys.version.splitlines()[0])
print('platform:', platform.platform())
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('scikit-learn:', sklearn.__version__)

In [ ]:
# Cell 2: Configuration - update MODEL_FILE if needed
OUTPUT_DIR = 'trained_models'
# If you know the exact filename, replace MODEL_FILE with it. Otherwise the notebook will try to find the most recent joblib file.
MODEL_FILE = None  # e.g. 'best_pipeline_random_forest_20251009T123456Z.joblib' or None to auto-detect
print('Will load artifacts from', OUTPUT_DIR)

In [ ]:
# Cell 3: Helpers - find the pipeline file if MODEL_FILE is None
import os, glob
if MODEL_FILE is None:
    candidates = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'best_pipeline_*.joblib')), reverse=True)
    if len(candidates) == 0:
        raise FileNotFoundError(f'No pipeline files found in {OUTPUT_DIR}. Please run the training notebook first.')
    MODEL_FILE = candidates[0]
print('Using model file:', MODEL_FILE)

In [ ]:
# Cell 4: Imports for evaluation & plotting
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd
import numpy as np


In [ ]:
# Cell 5: Load pipeline, metadata, and test data
pipeline = joblib.load(os.path.join(OUTPUT_DIR, MODEL_FILE))
print('Loaded pipeline:', pipeline)

meta_path = os.path.join(OUTPUT_DIR, 'training_metadata.json')
if os.path.exists(meta_path):
    import json
    with open(meta_path,'r') as f:
        metadata = json.load(f)
    print('Loaded metadata:', metadata.get('model_name', 'unknown'))
else:
    metadata = None
    print('No metadata found at', meta_path)

# Load test set
X_test_path = os.path.join(OUTPUT_DIR, 'X_test.csv')
y_test_path = os.path.join(OUTPUT_DIR, 'y_test.csv')
if not os.path.exists(X_test_path) or not os.path.exists(y_test_path):
    raise FileNotFoundError('X_test.csv or y_test.csv not found in trained_models/. Run training notebook or provide test files.')

X_test = pd.read_csv(X_test_path)
y_test = pd.read_csv(y_test_path).iloc[:, 0]
print('Loaded X_test shape:', X_test.shape, 'y_test shape:', y_test.shape)

In [ ]:
# Cell 6: Run predictions and compute metrics
y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

results = {
    'mae': float(mae),
    'mse': float(mse),
    'r2': float(r2)
}
print('Evaluation results:', results)

# Save results JSON
import json, os
with open(os.path.join(OUTPUT_DIR, 'test_evaluation_results.json'), 'w') as f:
    json.dump(results, f, indent=2)
print('Saved evaluation results to', os.path.join(OUTPUT_DIR, 'test_evaluation_results.json'))

# Save predictions CSV
pred_df = pd.DataFrame({'y_true': y_test, 'y_pred': y_pred})
pred_df.to_csv(os.path.join(OUTPUT_DIR, 'test_set_predictions_from_testing_notebook.csv'), index=False)
print('Saved predictions to', os.path.join(OUTPUT_DIR, 'test_set_predictions_from_testing_notebook.csv'))

In [ ]:
# Cell 7: Diagnostic plots - Predicted vs Actual and Residuals
import matplotlib.pyplot as plt
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Predicted vs Actual')
plt.show()

residuals = y_test - y_pred
plt.figure(figsize=(8,6))
plt.scatter(y_pred, residuals, alpha=0.6)
plt.axhline(0, color='k', linestyle='--')
plt.xlabel('Predicted')
plt.ylabel('Residuals')
plt.title('Residuals vs Predicted')
plt.show()

In [ ]:
# Cell 8: Error distribution and summary histogram
plt.figure(figsize=(8,5))
plt.hist(residuals, bins=40)
plt.title('Residuals distribution (y_true - y_pred)')
plt.xlabel('Residual')
plt.ylabel('Count')
plt.show()

# Quick numeric summary
print('Residuals mean:', residuals.mean())
print('Residuals std:', residuals.std())

In [ ]:
# Cell 9: Feature importance or coefficients (if model supports it)
def get_feature_names_from_preprocessor(preprocessor, numeric_cols, categorical_cols):
    names = []
    # numeric columns are passed through in order
    names.extend(numeric_cols)
    # categorical columns from one-hot encoder, if available
    if 'cat' in preprocessor.named_transformers_:
        ohe = preprocessor.named_transformers_['cat'].named_steps.get('onehot', None)
        if ohe is not None and hasattr(ohe, 'get_feature_names_out'):
            names.extend(list(ohe.get_feature_names_out(categorical_cols)))
    return names

# Attempt to extract feature names from metadata or pipeline
try:
    preproc = pipeline.named_steps.get('preprocessor', None)
    model = pipeline.named_steps.get('model', None)
    # Try to recover feature column lists from metadata if available
    if metadata and 'numeric_cols' in metadata and 'categorical_cols' in metadata:
        numeric_cols = metadata['numeric_cols']
        categorical_cols = metadata['categorical_cols']
    else:
        # Fallback: try to infer numeric/categorical from X_test
        numeric_cols = X_test.select_dtypes(include=['number']).columns.tolist()
        categorical_cols = X_test.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

    feat_names = get_feature_names_from_preprocessor(preproc, numeric_cols, categorical_cols)

    if model is not None and hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        fi = pd.Series(importances, index=feat_names).sort_values(ascending=False).head(30)
        display(fi)
    elif model is not None and hasattr(model, 'coef_'):
        coefs = model.coef_
        coef_series = pd.Series(coefs, index=feat_names).sort_values(key=abs, ascending=False).head(30)
        display(coef_series)
    else:
        print('No feature importance or coefficients available for this model.')
except Exception as e:
    print('Could not compute feature importances or coefficients:', e)

## Done

This testing notebook loaded the saved pipeline, ran predictions on `trained_models/X_test.csv`, saved `test_evaluation_results.json` and `test_set_predictions_from_testing_notebook.csv`, and displayed diagnostic plots.

If you'd like, I can automatically open the training or testing notebook for review or further customize plots (e.g., residual vs feature scatter, partial dependence, SHAP).